# Tome 0, Chapter 1: Sums, Binomials and Induction (Python companion)

This notebook accompanies Chapter 1 of *Order and Chance, Tome 0*.
The exercises come first, each with an empty cell for your code. Full solutions are in the
**Solutions** section at the end: try first, read after.

Libraries: `sympy`, `matplotlib` (see `requirements.txt` at the repository root).

In [ ]:
from fractions import Fraction
from math import comb, gcd
import sympy as sp
import matplotlib.pyplot as plt

## Exercises

**P1.1 (★) Test before you prove.** Check numerically that $\sum_{k=1}^n k^3 = \left(\frac{n(n+1)}{2}\right)^2$ for $n \le 1000$,
then let SymPy prove it symbolically with `sp.summation`.

In [ ]:
# Your code here


**P1.2 (★) Telescoping by machine.** Use `sp.apart` to decompose $\frac{1}{k(k+1)(k+2)}$ into partial fractions,
then compute $S_n = \sum_{k=1}^n \frac{1}{k(k+1)(k+2)}$ in closed form and its limit. Compare with Exercise 1.2 of the book.

In [ ]:
# Your code here


**P1.3 (★★) Pascal's triangle mod 2.** Draw the first 128 rows of Pascal's triangle, colouring $\binom{n}{k}$ black when it is odd.
Which famous fractal appears? (In Tome 1 you will prove why, with Lucas' theorem.)

In [ ]:
# Your code here


**P1.4 (★★) Hunting identities.** Write a function `check(lhs, rhs, N)` that tests an identity in $n$ for $n = 0, \dots, N$.
Use it on Vandermonde's identity with $m = n = r$, on $\sum_k k\binom{n}{k} = n2^{n-1}$, and on the hockey stick.

In [ ]:
# Your code here


**P1.5 (★★) CS application: the Fermat primality test.** Exercise 1.6 proves $a^p \equiv a \pmod p$ for $p$ prime.
1. Write `fermat_test(n, bases)` that declares $n$ "probably prime" if $a^{n} \equiv a \pmod n$ for every base given. Use Python's fast `pow(a, n, n)`.
2. Find all composite $n < 100\,000$ that pass the test for **every** base $a$ (the *Carmichael numbers*). Why does this make the Fermat test unsafe for cryptography? (Tome 1 replaces it with Miller–Rabin.)

In [ ]:
# Your code here


**P1.6 (★★) CS application: Fibonacci in $O(\log n)$.** From Exercise 1.7(a) derive the *fast doubling* formulas
$F_{2k} = F_k(2F_{k+1} - F_k)$ and $F_{2k+1} = F_k^2 + F_{k+1}^2$, implement them recursively, and compute $F_{10^6}$
(how many digits does it have?). Then check $\gcd(F_m, F_n) = F_{\gcd(m,n)}$ for all $m, n \le 60$.

In [ ]:
# Your code here


**P1.7 (★★★) Harmonic numbers are never integers.** Using exact `Fraction` arithmetic, compute $H_n$ for $n \le 200$.
Check that the denominator of $H_n$ is always divisible by $2^s$, where $2^s$ is the largest power of 2 not exceeding $n$
(this is exactly the proof of Exercise 1.9).

In [ ]:
# Your code here


---
## Solutions

### P1.1

In [ ]:
assert all(sum(k**3 for k in range(1, n + 1)) == (n * (n + 1) // 2) ** 2 for n in range(1001))

n, k = sp.symbols("n k", integer=True, positive=True)
lhs = sp.summation(k**3, (k, 1, n))
rhs = (n * (n + 1) / 2) ** 2
print(sp.factor(lhs), "  identity holds:", sp.simplify(lhs - rhs) == 0)

### P1.2

In [ ]:
term = 1 / (k * (k + 1) * (k + 2))
print("partial fractions:", sp.apart(term, k))
S = sp.simplify(sp.summation(term, (k, 1, n)))
print("S_n =", S)
print("limit =", sp.limit(S, n, sp.oo))
# The book's form 1/4 - 1/(2(n+1)(n+2)) agrees:
print(sp.simplify(S - (sp.Rational(1, 4) - 1 / (2 * (n + 1) * (n + 2)))) == 0)

### P1.3

In [ ]:
N = 128
grid = [[comb(r, c) % 2 if c <= r else 0 for c in range(N)] for r in range(N)]
plt.figure(figsize=(6, 6))
plt.imshow(grid, cmap="Greys", interpolation="nearest")
plt.title("Pascal's triangle mod 2: the Sierpiński triangle")
plt.axis("off")
plt.show()

The odd entries draw the **Sierpiński triangle**. Lucas' theorem says $\binom{n}{k}$ is odd exactly when every binary digit of $k$ is at most the corresponding digit of $n$, which is a self-similar condition.

### P1.4

In [ ]:
def check(lhs, rhs, N=60):
    bad = [n for n in range(N + 1) if lhs(n) != rhs(n)]
    return "OK" if not bad else f"fails at n = {bad[:5]}"

print("Vandermonde (m=n=r):", check(lambda n: sum(comb(n, k) ** 2 for k in range(n + 1)), lambda n: comb(2 * n, n)))
print("sum k C(n,k):       ", check(lambda n: sum(k * comb(n, k) for k in range(n + 1)), lambda n: n * 2 ** (n - 1) if n else 0))
print("hockey stick (r=3): ", check(lambda n: sum(comb(k, 3) for k in range(3, n + 1)), lambda n: comb(n + 1, 4)))

### P1.5

In [ ]:
def fermat_test(n, bases):
    return all(pow(a, n, n) == a % n for a in bases)

def is_prime(n):
    if n < 2:
        return False
    i = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += 1
    return True

# A composite n passes for every base exactly when a^n ≡ a (mod n) for all a in [0, n).
# Korselt's criterion makes this fast: n squarefree, and p - 1 | n - 1 for every prime p | n.
def is_carmichael(n):
    if n < 3 or is_prime(n):
        return False
    m, primes = n, []
    p = 2
    while p * p <= m:
        if m % p == 0:
            m //= p
            if m % p == 0:
                return False
            primes.append(p)
        p += 1
    if m > 1:
        primes.append(m)
    return len(primes) >= 2 and all((n - 1) % (p - 1) == 0 for p in primes)

carmichael = [n for n in range(2, 100_000) if is_carmichael(n)]
print(carmichael)
assert all(fermat_test(n, range(n)) for n in carmichael[:3])  # brute-force check of the first few

There are 16 Carmichael numbers below $10^5$, starting with $561 = 3 \cdot 11 \cdot 17$. They pass the Fermat test for every base,
so an adversary can choose one on purpose: a test that some composites always pass gives no guarantee, which is why cryptographic
libraries use Miller–Rabin instead.

### P1.6

In [ ]:
def fib_pair(n):
    """Return (F_n, F_{n+1}) by fast doubling."""
    if n == 0:
        return 0, 1
    a, b = fib_pair(n // 2)          # a = F_k, b = F_{k+1}, k = n // 2
    c = a * (2 * b - a)              # F_{2k}
    d = a * a + b * b                # F_{2k+1}
    return (d, c + d) if n % 2 else (c, d)

def fib(n):
    return fib_pair(n)[0]

assert [fib(i) for i in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
import sys
sys.set_int_max_str_digits(0)
print("F_(10^6) has", len(str(fib(10**6))), "digits")
assert all(gcd(fib(m), fib(n)) == fib(gcd(m, n)) for m in range(61) for n in range(61))
print("gcd(F_m, F_n) = F_gcd(m,n) checked for m, n <= 60")

The doubling formulas follow from Exercise 1.7(a): with $m = k+1$, $n = k$ we get $F_{2k+1} = F_{k+1}^2 + F_k^2$, and with $m = n = k$,
$F_{2k} = F_k F_{k+1} + F_{k-1}F_k = F_k(2F_{k+1} - F_k)$. Each step halves $n$, so only $O(\log n)$ multiplications are needed.

### P1.7

In [ ]:
H = Fraction(0)
for n in range(1, 201):
    H += Fraction(1, n)
    s = n.bit_length() - 1           # 2^s <= n < 2^(s+1)
    if n >= 2:
        assert H.denominator % 2**s == 0 and H.denominator > 1
print("For 2 <= n <= 200, H_n is never an integer; 2^s always divides its denominator.")
print("H_10 =", sum(Fraction(1, k) for k in range(1, 11)))